<a href="https://colab.research.google.com/github/maalaaikaa/Intro-to-Computer-Vision/blob/main/CV_Lab_01%20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Skin lesion classification
Transfer learning with ImageNet weights, fine-tuning, data augmentation, and dilated convolution.

In [1]:
!pip -q install kagglehub xgboost thop tabulate


In [2]:
import os, gc, random, tempfile, time
from pathlib import Path

import kagglehub
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import models, transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC, SVC
from xgboost import XGBClassifier
from thop import profile

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USING_GPU = DEVICE.type == "cuda"

# Change this to True only after enabling a Colab GPU.
FULL_EXPERIMENT = USING_GPU

IMAGE_SIZE = 224 if USING_GPU else 160
BATCH_SIZE = 24 if USING_GPU else 8
HEAD_EPOCHS = 2 if USING_GPU else 1
FINE_TUNE_EPOCHS = 4 if USING_GPU else 1
NUM_WORKERS = 2 if USING_GPU else 0

SELECTED_CLASSES = [
    "melanoma",
    "nevus",
    "basal cell carcinoma",
    "actinic keratosis",
]
LABEL_MAP = {name: i for i, name in enumerate(SELECTED_CLASSES)}
NUM_CLASSES = len(SELECTED_CLASSES)

MODEL_NAMES = (
    ["AlexNet", "ResNet50", "DenseNet121", "EfficientNet-B0"]
    if FULL_EXPERIMENT else ["AlexNet", "EfficientNet-B0"]
)

RESULT_DIR = Path("results_4class")
CHECKPOINT_DIR = Path("checkpoints_4class")
RESULT_DIR.mkdir(exist_ok=True)
CHECKPOINT_DIR.mkdir(exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if USING_GPU:
    torch.cuda.manual_seed_all(SEED)

print("Device:", DEVICE)
print("Models to run:", MODEL_NAMES)
print("Label mapping:", LABEL_MAP)


Device: cuda
Models to run: ['AlexNet', 'ResNet50', 'DenseNet121', 'EfficientNet-B0']
Label mapping: {'melanoma': 0, 'nevus': 1, 'basal cell carcinoma': 2, 'actinic keratosis': 3}


## Download dataset and create the four-class split

In [3]:
DATASET_ROOT = Path(kagglehub.dataset_download("nodoubttome/skin-cancer9-classesisic"))

def find_split_folder(root, split_name):
    candidates = [p for p in root.rglob("*") if p.is_dir() and p.name.lower() == split_name.lower()]
    if not candidates:
        raise FileNotFoundError(f"'{split_name}' folder was not found under {root}")
    return candidates[0]

TRAIN_DIR = find_split_folder(DATASET_ROOT, "train")
TEST_DIR = find_split_folder(DATASET_ROOT, "test")
print("Available classes:", sorted(p.name for p in TRAIN_DIR.iterdir() if p.is_dir()))

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def class_folder(split_dir, class_name):
    folders = {p.name.strip().lower(): p for p in split_dir.iterdir() if p.is_dir()}
    if class_name.lower() not in folders:
        raise ValueError(f"'{class_name}' is not available. Found: {sorted(folders)}")
    return folders[class_name.lower()]

class FourClassDataset(Dataset):
    def __init__(self, split_dir, class_names, transform):
        self.transform = transform
        self.samples = []
        for label, class_name in enumerate(class_names):
            paths = sorted(
                p for p in class_folder(split_dir, class_name).rglob("*")
                if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
            )
            self.samples.extend((p, label) for p in paths)
        self.targets = [label for _, label in self.samples]
        if not self.samples:
            raise RuntimeError("No images were found for the selected classes.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        path, label = self.samples[index]
        with Image.open(path) as image:
            image = image.convert("RGB")
        return self.transform(image), label

mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(12),
    transforms.ToTensor(), transforms.Normalize(mean, std),
])
eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(), transforms.Normalize(mean, std),
])

full_train_aug = FourClassDataset(TRAIN_DIR, SELECTED_CLASSES, train_transform)
full_train_eval = FourClassDataset(TRAIN_DIR, SELECTED_CLASSES, eval_transform)
test_dataset = FourClassDataset(TEST_DIR, SELECTED_CLASSES, eval_transform)

indices = np.arange(len(full_train_aug))
train_idx, val_idx = train_test_split(
    indices, test_size=0.20, random_state=SEED, stratify=full_train_aug.targets
)

train_dataset = Subset(full_train_aug, train_idx)
val_dataset = Subset(full_train_eval, val_idx)
train_eval_dataset = Subset(full_train_eval, train_idx)

loader_kwargs = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=USING_GPU)
train_loader = DataLoader(train_dataset, shuffle=True, **loader_kwargs)
val_loader = DataLoader(val_dataset, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_kwargs)
train_eval_loader = DataLoader(train_eval_dataset, shuffle=False, **loader_kwargs)

def counts(labels):
    return np.bincount(labels, minlength=NUM_CLASSES)

train_counts = counts([full_train_aug.targets[i] for i in train_idx])
val_counts = counts([full_train_aug.targets[i] for i in val_idx])
test_counts = counts(test_dataset.targets)

label_table = pd.DataFrame({
    "Label": range(NUM_CLASSES), "Class Name": SELECTED_CLASSES,
    "Training Images": train_counts, "Validation Images": val_counts,
    "Testing Images": test_counts,
})
display(label_table)
label_table.to_csv(RESULT_DIR / "label_table.csv", index=False)


Using Colab cache for faster access to the 'skin-cancer9-classesisic' dataset.
Available classes: ['actinic keratosis', 'basal cell carcinoma', 'dermatofibroma', 'melanoma', 'nevus', 'pigmented benign keratosis', 'seborrheic keratosis', 'squamous cell carcinoma', 'vascular lesion']


,Label,Class Name,Training Images,Validation Images,Testing Images
0,0,melanoma,350,88,16
1,1,nevus,286,71,16
2,2,basal cell carcinoma,301,75,16
3,3,actinic keratosis,91,23,16


## Model, training, and evaluation functions

In [4]:
def build_model(model_name):
    if model_name == "AlexNet":
        model = models.alexnet(weights=models.AlexNet_Weights.DEFAULT)
        model.classifier[6] = nn.Linear(model.classifier[6].in_features, NUM_CLASSES)
    elif model_name == "ResNet50":
        model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
    elif model_name == "DenseNet121":
        model = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
        model.classifier = nn.Linear(model.classifier.in_features, NUM_CLASSES)
    elif model_name == "EfficientNet-B0":
        model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASSES)
    else:
        raise ValueError(f"Unsupported model: {model_name}")
    return model

def head_parameters(model_name, model):
    if model_name in ["AlexNet", "EfficientNet-B0"]:
        return model.classifier.parameters()
    if model_name == "DenseNet121":
        return model.classifier.parameters()
    return model.fc.parameters()

@torch.no_grad()
def evaluate_model(model, loader):
    model.eval()
    y_true, y_pred, y_prob = [], [], []
    for images, labels in loader:
        probabilities = torch.softmax(model(images.to(DEVICE)), dim=1)
        y_true.extend(labels.numpy())
        y_pred.extend(probabilities.argmax(dim=1).cpu().numpy())
        y_prob.extend(probabilities.cpu().numpy())

    y_true, y_pred, y_prob = np.array(y_true), np.array(y_pred), np.array(y_prob)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    try:
        auc = roc_auc_score(y_true, y_prob, multi_class="ovr", average="macro")
    except ValueError:
        auc = np.nan
    return {
        "Accuracy (%)": 100 * accuracy_score(y_true, y_pred),
        "Precision (%)": 100 * precision,
        "Recall (%)": 100 * recall,
        "F1-Score (%)": 100 * f1,
        "AUC (%)": 100 * auc,
    }

def fit_epochs(model, optimizer, criterion, epochs, model_name, stage, best_accuracy, checkpoint):
    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        metrics = evaluate_model(model, val_loader)
        print(f"{model_name} | {stage} {epoch + 1}/{epochs} | loss={total_loss / len(train_loader):.4f} | val_acc={metrics['Accuracy (%)']:.2f}%")
        if metrics["Accuracy (%)"] > best_accuracy:
            best_accuracy = metrics["Accuracy (%)"]
            torch.save(model.state_dict(), checkpoint)
    return best_accuracy

def train_model(model_name):
    model = build_model(model_name).to(DEVICE)
    class_counts = np.bincount([full_train_aug.targets[i] for i in train_idx], minlength=NUM_CLASSES)
    class_weights = torch.tensor(len(train_idx) / (NUM_CLASSES * class_counts), dtype=torch.float32, device=DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    checkpoint = CHECKPOINT_DIR / f"{model_name.replace('-', '_')}.pth"
    best_accuracy = -1.0

    # Fast transfer-learning stage: train only the new classifier head.
    for parameter in model.parameters():
        parameter.requires_grad = False
    for parameter in head_parameters(model_name, model):
        parameter.requires_grad = True
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)
    best_accuracy = fit_epochs(model, optimizer, criterion, HEAD_EPOCHS, model_name, "head", best_accuracy, checkpoint)

    # Fine-tune the complete network with a smaller learning rate.
    for parameter in model.parameters():
        parameter.requires_grad = True
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
    best_accuracy = fit_epochs(model, optimizer, criterion, FINE_TUNE_EPOCHS, model_name, "fine-tune", best_accuracy, checkpoint)

    model.load_state_dict(torch.load(checkpoint, map_location=DEVICE))
    return model, evaluate_model(model, test_loader), checkpoint


In [5]:
# Add remaining models to build_model()

def build_model(model_name):
    if model_name == "AlexNet":
        model = models.alexnet(weights=models.AlexNet_Weights.DEFAULT)
        model.classifier[6] = nn.Linear(model.classifier[6].in_features, NUM_CLASSES)

    elif model_name == "VGG16":
        model = models.vgg16(weights=models.VGG16_Weights.DEFAULT)
        model.classifier[6] = nn.Linear(model.classifier[6].in_features, NUM_CLASSES)

    elif model_name == "VGG19":
        model = models.vgg19(weights=models.VGG19_Weights.DEFAULT)
        model.classifier[6] = nn.Linear(model.classifier[6].in_features, NUM_CLASSES)

    elif model_name == "ResNet18":
        model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)

    elif model_name == "ResNet50":
        model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)

    elif model_name == "ResNet101":
        model = models.resnet101(weights=models.ResNet101_Weights.DEFAULT)
        model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)

    elif model_name == "DenseNet121":
        model = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
        model.classifier = nn.Linear(model.classifier.in_features, NUM_CLASSES)

    elif model_name == "EfficientNet-B0":
        model = models.efficientnet_b0(
            weights=models.EfficientNet_B0_Weights.DEFAULT
        )
        model.classifier[1] = nn.Linear(
            model.classifier[1].in_features,
            NUM_CLASSES
        )

    else:
        raise ValueError(f"Invalid model name: {model_name}")

    return model


# This function trains, tests, and saves one model result.
def run_one_model(model_name):
    global table1_rows, checkpoint_paths

    if "table1_rows" not in globals():
        table1_rows = []

    if "checkpoint_paths" not in globals():
        checkpoint_paths = {}

    print(f"\nTraining {model_name}...")

    model, metrics, checkpoint_path = train_model(model_name)

    # Remove old duplicate result, if present
    table1_rows = [
        row for row in table1_rows
        if row["Model"] != model_name
    ]

    table1_rows.append({
        "Model": model_name,
        **metrics
    })

    checkpoint_paths[model_name] = checkpoint_path

    print(f"\n{model_name} Test Results:")
    print(metrics)

    del model
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## Table 1 — Transfer-learning models

In [6]:
table1_rows, checkpoint_paths = [], {}
for model_name in MODEL_NAMES:
    print(f"\nTraining {model_name}...")
    model, metrics, checkpoint = train_model(model_name)
    table1_rows.append({"Model": model_name, **metrics})
    checkpoint_paths[model_name] = checkpoint
    del model
    gc.collect()
    if USING_GPU:
        torch.cuda.empty_cache()

table1 = pd.DataFrame(table1_rows).round(2)
display(table1)
table1.to_csv(RESULT_DIR / "table1_transfer_models.csv", index=False)



Training AlexNet...
Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100%|██████████| 233M/233M [00:02<00:00, 89.9MB/s]


AlexNet | head 1/2 | loss=1.3181 | val_acc=68.87%
AlexNet | head 2/2 | loss=0.9712 | val_acc=69.26%
AlexNet | fine-tune 1/4 | loss=0.7818 | val_acc=72.37%
AlexNet | fine-tune 2/4 | loss=0.5920 | val_acc=78.21%
AlexNet | fine-tune 3/4 | loss=0.5199 | val_acc=81.32%
AlexNet | fine-tune 4/4 | loss=0.4865 | val_acc=76.26%

Training ResNet50...
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 172MB/s]


ResNet50 | head 1/2 | loss=1.1959 | val_acc=66.93%
ResNet50 | head 2/2 | loss=0.9472 | val_acc=70.04%
ResNet50 | fine-tune 1/4 | loss=0.7621 | val_acc=77.82%
ResNet50 | fine-tune 2/4 | loss=0.4755 | val_acc=81.32%
ResNet50 | fine-tune 3/4 | loss=0.4077 | val_acc=81.71%
ResNet50 | fine-tune 4/4 | loss=0.3075 | val_acc=77.82%

Training DenseNet121...
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 162MB/s]


DenseNet121 | head 1/2 | loss=1.2564 | val_acc=55.64%
DenseNet121 | head 2/2 | loss=0.9916 | val_acc=70.82%
DenseNet121 | fine-tune 1/4 | loss=0.7413 | val_acc=77.43%
DenseNet121 | fine-tune 2/4 | loss=0.4826 | val_acc=80.54%
DenseNet121 | fine-tune 3/4 | loss=0.3916 | val_acc=81.71%
DenseNet121 | fine-tune 4/4 | loss=0.3187 | val_acc=83.27%

Training EfficientNet-B0...
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 153MB/s]


EfficientNet-B0 | head 1/2 | loss=1.1870 | val_acc=67.32%
EfficientNet-B0 | head 2/2 | loss=0.9417 | val_acc=68.48%
EfficientNet-B0 | fine-tune 1/4 | loss=0.7866 | val_acc=74.71%
EfficientNet-B0 | fine-tune 2/4 | loss=0.6018 | val_acc=78.99%
EfficientNet-B0 | fine-tune 3/4 | loss=0.4989 | val_acc=82.10%
EfficientNet-B0 | fine-tune 4/4 | loss=0.4078 | val_acc=82.88%


,Model,Accuracy (%),Precision (%),Recall (%),F1-Score (%),AUC (%)
0,AlexNet,65.62,68.56,65.62,63.95,90.17
1,ResNet50,62.50,66.50,62.50,57.41,87.37
2,DenseNet121,70.31,76.82,70.31,68.99,92.84
3,EfficientNet-B0,78.12,82.05,78.12,76.95,93.82


In [7]:
def head_parameters(model_name, model):
    if model_name in [
        "AlexNet",
        "VGG16",
        "VGG19",
        "EfficientNet-B0"
    ]:
        return model.classifier.parameters()

    elif model_name == "DenseNet121":
        return model.classifier.parameters()

    elif model_name in [
        "ResNet18",
        "ResNet50",
        "ResNet101"
    ]:
        return model.fc.parameters()

    else:
        raise ValueError(f"Invalid model name: {model_name}")

In [8]:
def head_parameters(model_name, model):
    if model_name in [
        "AlexNet",
        "VGG16",
        "VGG19",
        "EfficientNet-B0"
    ]:
        return model.classifier.parameters()

    elif model_name == "DenseNet121":
        return model.classifier.parameters()

    elif model_name in [
        "ResNet18",
        "ResNet50",
        "ResNet101"
    ]:
        return model.fc.parameters()

    else:
        raise ValueError(f"Invalid model name: {model_name}")

In [9]:
run_one_model("VGG16")


Training VGG16...
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:03<00:00, 171MB/s]


VGG16 | head 1/2 | loss=1.8572 | val_acc=66.93%
VGG16 | head 2/2 | loss=1.2474 | val_acc=71.98%
VGG16 | fine-tune 1/4 | loss=0.9557 | val_acc=71.21%
VGG16 | fine-tune 2/4 | loss=0.7644 | val_acc=77.82%
VGG16 | fine-tune 3/4 | loss=0.6336 | val_acc=77.82%
VGG16 | fine-tune 4/4 | loss=0.5997 | val_acc=72.76%

VGG16 Test Results:
{'Accuracy (%)': 67.1875, 'Precision (%)': 75.26515151515152, 'Recall (%)': 67.1875, 'F1-Score (%)': 62.61060259344012, 'AUC (%)': np.float64(90.78776041666667)}


In [10]:
run_one_model("VGG19")


Training VGG19...
Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to /root/.cache/torch/hub/checkpoints/vgg19-dcbb9e9d.pth


100%|██████████| 548M/548M [00:03<00:00, 146MB/s]


VGG19 | head 1/2 | loss=1.8812 | val_acc=48.64%
VGG19 | head 2/2 | loss=1.4143 | val_acc=40.47%
VGG19 | fine-tune 1/4 | loss=1.2955 | val_acc=68.48%
VGG19 | fine-tune 2/4 | loss=0.8999 | val_acc=67.32%
VGG19 | fine-tune 3/4 | loss=0.8325 | val_acc=75.88%
VGG19 | fine-tune 4/4 | loss=0.8190 | val_acc=75.88%

VGG19 Test Results:
{'Accuracy (%)': 76.5625, 'Precision (%)': 78.0734004127967, 'Recall (%)': 76.5625, 'F1-Score (%)': 76.75595238095238, 'AUC (%)': np.float64(91.73177083333334)}


In [11]:
run_one_model("ResNet18")


Training ResNet18...
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 166MB/s]


ResNet18 | head 1/2 | loss=1.2064 | val_acc=60.70%
ResNet18 | head 2/2 | loss=0.9556 | val_acc=61.48%
ResNet18 | fine-tune 1/4 | loss=0.7367 | val_acc=76.65%
ResNet18 | fine-tune 2/4 | loss=0.4655 | val_acc=79.38%
ResNet18 | fine-tune 3/4 | loss=0.3834 | val_acc=80.16%
ResNet18 | fine-tune 4/4 | loss=0.3482 | val_acc=80.54%

ResNet18 Test Results:
{'Accuracy (%)': 64.0625, 'Precision (%)': 69.62121212121212, 'Recall (%)': 64.0625, 'F1-Score (%)': 62.07346483743999, 'AUC (%)': np.float64(88.8671875)}


In [12]:
run_one_model("ResNet50")


Training ResNet50...
ResNet50 | head 1/2 | loss=1.2117 | val_acc=68.87%
ResNet50 | head 2/2 | loss=0.9582 | val_acc=70.04%
ResNet50 | fine-tune 1/4 | loss=0.7279 | val_acc=79.77%
ResNet50 | fine-tune 2/4 | loss=0.4762 | val_acc=81.71%
ResNet50 | fine-tune 3/4 | loss=0.3646 | val_acc=80.93%
ResNet50 | fine-tune 4/4 | loss=0.3001 | val_acc=84.44%

ResNet50 Test Results:
{'Accuracy (%)': 67.1875, 'Precision (%)': 74.32386747802569, 'Recall (%)': 67.1875, 'F1-Score (%)': 66.76767676767676, 'AUC (%)': np.float64(89.6484375)}


In [13]:
run_one_model("ResNet101")


Training ResNet101...
Downloading: "https://download.pytorch.org/models/resnet101-cd907fc2.pth" to /root/.cache/torch/hub/checkpoints/resnet101-cd907fc2.pth


100%|██████████| 171M/171M [00:01<00:00, 137MB/s]


ResNet101 | head 1/2 | loss=1.1694 | val_acc=63.04%
ResNet101 | head 2/2 | loss=0.9491 | val_acc=63.42%
ResNet101 | fine-tune 1/4 | loss=0.7328 | val_acc=76.26%
ResNet101 | fine-tune 2/4 | loss=0.4829 | val_acc=79.38%
ResNet101 | fine-tune 3/4 | loss=0.3688 | val_acc=81.71%
ResNet101 | fine-tune 4/4 | loss=0.2830 | val_acc=80.93%

ResNet101 Test Results:
{'Accuracy (%)': 67.1875, 'Precision (%)': 71.64426977687627, 'Recall (%)': 67.1875, 'F1-Score (%)': 65.60314685314685, 'AUC (%)': np.float64(93.26171875)}


In [14]:
run_one_model("DenseNet121")


Training DenseNet121...
DenseNet121 | head 1/2 | loss=1.2097 | val_acc=66.93%
DenseNet121 | head 2/2 | loss=0.9406 | val_acc=64.20%
DenseNet121 | fine-tune 1/4 | loss=0.7309 | val_acc=80.16%
DenseNet121 | fine-tune 2/4 | loss=0.4833 | val_acc=80.16%
DenseNet121 | fine-tune 3/4 | loss=0.3733 | val_acc=80.93%
DenseNet121 | fine-tune 4/4 | loss=0.3175 | val_acc=78.99%

DenseNet121 Test Results:
{'Accuracy (%)': 71.875, 'Precision (%)': 81.2609649122807, 'Recall (%)': 71.875, 'F1-Score (%)': 70.04917184265011, 'AUC (%)': np.float64(90.39713541666666)}


In [15]:
import pandas as pd

model_order = [
    "AlexNet",
    "VGG16",
    "VGG19",
    "ResNet18",
    "ResNet50",
    "ResNet101",
    "DenseNet121",
    "EfficientNet-B0"
]

table1 = pd.DataFrame(table1_rows).round(2)

table1["Model"] = pd.Categorical(
    table1["Model"],
    categories=model_order,
    ordered=True
)

table1 = table1.sort_values("Model")

display(table1)

table1.to_csv(
    RESULT_DIR / "table1_transfer_learning_models.csv",
    index=False
)

,Model,Accuracy (%),Precision (%),Recall (%),F1-Score (%),AUC (%)
0,AlexNet,65.62,68.56,65.62,63.95,90.17
2,VGG16,67.19,75.27,67.19,62.61,90.79
3,VGG19,76.56,78.07,76.56,76.76,91.73
4,ResNet18,64.06,69.62,64.06,62.07,88.87
5,ResNet50,67.19,74.32,67.19,66.77,89.65
6,ResNet101,67.19,71.64,67.19,65.60,93.26
7,DenseNet121,71.88,81.26,71.88,70.05,90.40
1,EfficientNet-B0,78.12,82.05,78.12,76.95,93.82


## Table 2 — Classical classifiers using EfficientNet-B0 deep features

In [16]:
def feature_extractor():
    model = build_model("EfficientNet-B0")
    model.load_state_dict(torch.load(checkpoint_paths["EfficientNet-B0"], map_location=DEVICE))
    model.classifier = nn.Identity()
    return model.to(DEVICE).eval()

@torch.no_grad()
def extract_features(model, loader):
    feature_list, label_list = [], []
    for images, labels in loader:
        feature_list.append(model(images.to(DEVICE)).cpu().numpy())
        label_list.append(labels.numpy())
    return np.concatenate(feature_list), np.concatenate(label_list)

extractor = feature_extractor()
X_train, y_train = extract_features(extractor, train_eval_loader)
X_test, y_test = extract_features(extractor, test_loader)
del extractor
gc.collect()
if USING_GPU:
    torch.cuda.empty_cache()

classifiers = {
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED)),
    "Decision Tree": DecisionTreeClassifier(class_weight="balanced", random_state=SEED),
    "Random Forest": RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=SEED, n_jobs=-1),
    "K-Nearest Neighbors (KNN)": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5, weights="distance")),
    "Linear SVM": make_pipeline(StandardScaler(), LinearSVC(class_weight="balanced", random_state=SEED)),
    "RBF-SVM": make_pipeline(StandardScaler(), SVC(kernel="rbf", probability=True, class_weight="balanced", random_state=SEED)),
    "XGBoost": XGBClassifier(n_estimators=150, max_depth=4, learning_rate=0.05, subsample=0.85, colsample_bytree=0.85, objective="multi:softprob", num_class=NUM_CLASSES, eval_metric="mlogloss", random_state=SEED, n_jobs=-1),
}

def probabilities_for(classifier, X):
    if hasattr(classifier, "predict_proba"):
        return classifier.predict_proba(X)
    scores = classifier.decision_function(X)
    scores -= scores.max(axis=1, keepdims=True)
    scores = np.exp(scores)
    return scores / scores.sum(axis=1, keepdims=True)

table2_rows = []
for name, classifier in classifiers.items():
    classifier.fit(X_train, y_train)
    predictions = classifier.predict(X_test)
    probabilities = probabilities_for(classifier, X_test)
    precision, recall, f1, _ = precision_recall_fscore_support(y_test, predictions, average="macro", zero_division=0)
    try:
        auc = roc_auc_score(y_test, probabilities, multi_class="ovr", average="macro")
    except ValueError:
        auc = np.nan
    table2_rows.append({"Feature Extractor": "EfficientNet-B0 Deep Features", "Classifier": name, "Accuracy (%)": 100 * accuracy_score(y_test, predictions), "Precision (%)": 100 * precision, "Recall (%)": 100 * recall, "F1-Score (%)": 100 * f1, "AUC (%)": 100 * auc})

table2 = pd.DataFrame(table2_rows).round(2)
display(table2)
table2.to_csv(RESULT_DIR / "table2_classifiers.csv", index=False)


/usr/local/lib/python3.13/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


,Feature Extractor,Classifier,Accuracy (%),Precision (%),Recall (%),F1-Score (%),AUC (%)
0,EfficientNet-B0 Deep Features,Logistic Regression,59.38,78.31,59.38,52.38,91.70
1,EfficientNet-B0 Deep Features,Decision Tree,53.12,55.07,53.12,50.10,69.16
2,EfficientNet-B0 Deep Features,Random Forest,60.94,53.74,60.94,52.53,90.92
3,EfficientNet-B0 Deep Features,K-Nearest Neighbors (KNN),57.81,75.20,57.81,50.56,86.60
4,EfficientNet-B0 Deep Features,Linear SVM,62.50,79.46,62.50,57.22,86.82
5,EfficientNet-B0 Deep Features,RBF-SVM,73.44,81.98,73.44,71.96,92.09
6,EfficientNet-B0 Deep Features,XGBoost,54.69,48.75,54.69,44.01,90.17


## Table 3 — Computational efficiency

In [17]:
import torch

def model_size_mb(model):
    with tempfile.NamedTemporaryFile(suffix=".pth") as file:
        torch.save(model.state_dict(), file.name)
        return os.path.getsize(file.name) / (1024 ** 2)

@torch.no_grad()
def inference_time_ms(model, repetitions=20, warmups=5):
    model.eval()
    image = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE)
    for _ in range(warmups):
        model(image)
    if USING_GPU:
        torch.cuda.synchronize()
    start = time.perf_counter()
    for _ in range(repetitions):
        model(image)
    if USING_GPU:
        torch.cuda.synchronize()
    return 1000 * (time.perf_counter() - start) / repetitions

accuracy_lookup = table1.set_index("Model")["Accuracy (%)"].to_dict()
table3_rows = []
for model_name in MODEL_NAMES:
    model = build_model(model_name)
    model.load_state_dict(torch.load(checkpoint_paths[model_name], map_location="cpu"))
    parameters_m = sum(p.numel() for p in model.parameters()) / 1e6
    size_mb = model_size_mb(model)
    flops, _ = profile(model, inputs=(torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE),), verbose=False)
    model = model.to(DEVICE)
    table3_rows.append({"Model": model_name, "Parameters (M)": parameters_m, "Model Size (MB)": size_mb, "FLOPs (G)": flops / 1e9, "Inference Time (ms)": inference_time_ms(model), "Accuracy (%)": accuracy_lookup[model_name]})
    del model
    gc.collect()
    if USING_GPU:
        torch.cuda.empty_cache()

table3 = pd.DataFrame(table3_rows).round(2)
display(table3)
table3.to_csv(RESULT_DIR / "table3_efficiency.csv", index=False)

,Model,Parameters (M),Model Size (MB),FLOPs (G),Inference Time (ms),Accuracy (%)
0,AlexNet,57.02,217.52,0.71,2.11,65.62
1,ResNet50,23.52,90.01,4.13,9.74,67.19
2,DenseNet121,6.96,27.11,2.90,13.77,71.88
3,EfficientNet-B0,4.01,15.59,0.41,8.02,78.12


In [21]:
print("Best transfer-learning model:")
display(table1.loc[[table1["Accuracy (%)"].idxmax()]])
print("Best classical classifier:")
display(table2.loc[[table2["Accuracy (%)"].idxmax()]])
print("CSV result files are saved in:", RESULT_DIR)


Best transfer-learning model:


,Model,Accuracy (%),Precision (%),Recall (%),F1-Score (%),AUC (%)
1,EfficientNet-B0,78.12,82.05,78.12,76.95,93.82


Best classical classifier:


,Feature Extractor,Classifier,Accuracy (%),Precision (%),Recall (%),F1-Score (%),AUC (%)
5,EfficientNet-B0 Deep Features,RBF-SVM,73.44,81.98,73.44,71.96,92.09


CSV result files are saved in: results_4class


In [22]:
output_file = RESULT_DIR / "output.md"